# Fase 3 -- Construcción de métricas de acceso

Orquesta `src/metrics.py`: cada métrica es una función que toma un
DataFrame y devuelve un DataFrame (nada de lógica de métricas vive en el
dashboard). Este notebook solo llama funciones, imprime resultados y
exporta a `data/outputs/` -- las tablas exactas que usa el reporte
LaTeX (Fase 5) y el dashboard (Fase 4).

In [1]:
import os
import sys

import geopandas as gpd
import pandas as pd

BASE = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "data" else os.getcwd()
sys.path.insert(0, BASE)
from src import metrics as M
from src import routing as R

PROCESSED_DIR = os.path.join(BASE, "data", "processed")
OUTPUTS_DIR = os.path.join(BASE, "data", "outputs")

R.log("=== FASE 3: METRICAS ===")


[17:29:39] === FASE 3: METRICAS ===


## 1. Tabla de acceso por punto de demanda

Une la muestra de demanda (Fase 2, con `POBLACION_CP` y `ES_URBANO`) con el
tiempo de acceso en auto al resolutivo más cercano (`nearest_car.parquet`).
Los puntos sin ruta se mantienen con `time_min_car = NaN` y
`ALCANZABLE = False` -- no se descartan ni se imputan con 0.

In [2]:
demanda = gpd.read_parquet(os.path.join(PROCESSED_DIR, "demanda_muestra.parquet"))
nearest_car = pd.read_parquet(os.path.join(PROCESSED_DIR, "nearest_car.parquet"))

df = demanda.merge(nearest_car, on="DEMAND_ID", how="left")
df = df.rename(columns={
    "DIST": "DISTRITO", "PROV": "PROVINCIA", "DEP_NORM": "DEPARTAMENTO",
    "time_min": "time_min_car",
})

acceso = M.tabla_acceso(df)
acceso.to_csv(os.path.join(OUTPUTS_DIR, "tabla_acceso.csv"), index=False)
R.log(f"tabla_acceso: {len(acceso)} filas, {acceso['ALCANZABLE'].mean():.1%} alcanzables por auto")


[17:29:39] tabla_acceso: 5000 filas, 71.2% alcanzables por auto


## 2. Cobertura por bandas (30 / 60 / 120 min, y sin ruta)

Porcentaje de **población** (no de centros poblados) dentro de cada banda.
Un punto sin ruta cuenta aparte (`SIN_RUTA`), no se mezcla con `> 120 min`
-- son dos problemas distintos: "tarda mucho" vs. "no hay dato de ruta".

In [3]:
bandas_general = M.bandas_cobertura(acceso)
bandas_general.to_csv(os.path.join(OUTPUTS_DIR, "cobertura_bandas_general.csv"), index=False)
print(bandas_general)

bandas_depto = M.bandas_cobertura(acceso, by=["DEPARTAMENTO"])
bandas_depto.to_csv(os.path.join(OUTPUTS_DIR, "cobertura_bandas_departamento.csv"), index=False)
print(bandas_depto)


        banda     poblacion  poblacion_total_grupo  pct_poblacion
0  <= 120 min  2.538829e+05              3738681.0       0.067907
1   <= 30 min  2.058445e+06              3738681.0       0.550580
2   <= 60 min  5.150622e+05              3738681.0       0.137766
3   > 120 min  1.058835e+05              3738681.0       0.028321
4    SIN_RUTA  8.054076e+05              3738681.0       0.215426
   DEPARTAMENTO       banda     poblacion  poblacion_total_grupo  \
0         JUNIN  <= 120 min  1.250914e+05              1379934.0   
1         JUNIN   <= 30 min  8.809688e+05              1379934.0   
2         JUNIN   <= 60 min  2.734086e+05              1379934.0   
3         JUNIN   > 120 min  5.433560e+04              1379934.0   
4         JUNIN    SIN_RUTA  4.612959e+04              1379934.0   
5    LAMBAYEQUE  <= 120 min  7.713671e+04              1290617.0   
6    LAMBAYEQUE   <= 30 min  1.019570e+06              1290617.0   
7    LAMBAYEQUE   <= 60 min  1.440516e+05              12906

## 3. Acceso ponderado por población -- distrito / provincia / departamento

Promedio de tiempo de acceso ponderado por `POBLACION_CP` en cada nivel
administrativo. Ponderar importa: un caserío de 12 personas no debe pesar
igual que un pueblo de 12,000 en el promedio distrital.

In [4]:
acceso_distrital = M.acceso_ponderado_por_nivel(acceso, "DISTRITO")
acceso_provincial = M.acceso_ponderado_por_nivel(acceso, "PROVINCIA")
acceso_departamental = M.acceso_ponderado_por_nivel(acceso, "DEPARTAMENTO")

mapa_dep = acceso[["UBIGEO", "DEPARTAMENTO"]].drop_duplicates()
acceso_distrital = acceso_distrital.merge(mapa_dep, on="UBIGEO", how="left")

acceso_distrital.to_csv(os.path.join(OUTPUTS_DIR, "acceso_distrital.csv"), index=False)
acceso_provincial.to_csv(os.path.join(OUTPUTS_DIR, "acceso_provincial.csv"), index=False)
acceso_departamental.to_csv(os.path.join(OUTPUTS_DIR, "acceso_departamental.csv"), index=False)
R.log(f"Acceso ponderado -- distritos: {len(acceso_distrital)}, provincias: {len(acceso_provincial)}, "
      f"departamentos: {len(acceso_departamental)}")
print(acceso_departamental)


[17:29:40] Acceso ponderado -- distritos: 215, provincias: 20, departamentos: 3


  DEPARTAMENTO  poblacion_total  tiempo_medio_ponderado_min  pct_sin_ruta  \
0       LORETO        1068130.0                   34.993183      0.895442   
1        JUNIN        1379934.0                   32.909262      0.036226   
2   LAMBAYEQUE        1290617.0                   22.699704      0.006993   

   n_centros_poblados  
0                1492  
1                2650  
2                 858  


## 4. Lista de brechas críticas (peores 15 distritos)

In [5]:
brechas = M.lista_brechas_criticas(acceso_distrital, n=15)
brechas.to_csv(os.path.join(OUTPUTS_DIR, "brechas_criticas.csv"), index=False)
print(brechas[["DISTRITO", "DEPARTAMENTO", "tiempo_medio_ponderado_min", "poblacion_total"]].to_string())


                        DISTRITO DEPARTAMENTO  tiempo_medio_ponderado_min  poblacion_total
0                  PAMPA HERMOSA       LORETO                  279.917620          11081.0
1               VIZCATAN DEL ENE        JUNIN                  232.517661           3573.0
2                      RIO TAMBO        JUNIN                  228.191416          61259.0
3                        CAÑARIS   LAMBAYEQUE                  197.952536          14787.0
4      SANTO DOMINGO DE ACOBAMBA        JUNIN                  176.828286           7776.0
5                      INCAHUASI   LAMBAYEQUE                  144.178486          15733.0
6                      ANDAMARCA        JUNIN                  125.927903           4536.0
7     TENIENTE CESAR LOPEZ ROJAS       LORETO                  117.643448           6743.0
8                          OLMOS   LAMBAYEQUE                  113.893301          41587.0
9                          SALAS   LAMBAYEQUE                  112.379634          13056.0

## 5. Desigualdad de acceso -- Gini ponderado y curva de Lorenz

Se elige el coeficiente de Gini (ver justificación en el docstring de
`gini_ponderado`): acotado en [0,1], interpretable, estándar en estudios de
accesibilidad en salud, y se pondera naturalmente por población.

In [6]:
gini_global = M.gini_ponderado(acceso["time_min_car"], acceso["POBLACION_CP"])
gini_por_depto = {
    dep: M.gini_ponderado(g["time_min_car"], g["POBLACION_CP"])
    for dep, g in acceso.groupby("DEPARTAMENTO")
}
pd.DataFrame(
    [{"nivel": "3_DEPARTAMENTOS", "gini": gini_global}] +
    [{"nivel": k, "gini": v} for k, v in gini_por_depto.items()]
).to_csv(os.path.join(OUTPUTS_DIR, "gini_acceso.csv"), index=False)
R.log(f"Gini global de acceso (3 deptos): {gini_global:.3f}")
print(gini_por_depto)

lorenz = M.curva_lorenz(acceso["time_min_car"], acceso["POBLACION_CP"])
lorenz.to_csv(os.path.join(OUTPUTS_DIR, "lorenz_acceso.csv"), index=False)


[17:29:40] Gini global de acceso (3 deptos): 0.633


{'JUNIN': 0.6318173725727605, 'LAMBAYEQUE': 0.6465189274178113, 'LORETO': 0.5177705733732545}


## 6. Urbano vs. rural

Regla explícita: **urbano** = el centro poblado es capital de distrito,
provincia o departamento (columna `CAPITAL != 0` en SIGMED). Es una regla
administrativa, no de densidad real -- se declara así en el reporte.

In [7]:
urb_rural = M.contraste_urbano_rural(acceso)
urb_rural.to_csv(os.path.join(OUTPUTS_DIR, "urbano_rural.csv"), index=False)
print(urb_rural)


    clase  n_centros_poblados  poblacion_total  tiempo_medio_ponderado_min  \
0   rural                4918     3.664972e+06                   29.047781   
1  urbano                  82     7.370892e+04                   12.109374   

   pct_sin_ruta  
0      0.289345  
1      0.182927  


## 7. Cruce con densidad poblacional distrital (cross-analysis obligatorio)

Se cruza el acceso ponderado distrital con la densidad poblacional
(Población/Superficie, INEI) del mismo distrito.

**Interpretación (obligatoria declararla así):** la densidad es un *proxy*
de ruralidad y aislamiento geográfico -- no una causa directa del tiempo de
acceso. La relación reportada es **correlacional, no causal**: densidad y
acceso comparten causas comunes (geografía, historia de inversión pública)
que este análisis no puede aislar.

In [8]:
pop_completa = pd.read_csv(os.path.join(OUTPUTS_DIR, "poblacion_distrital_inei.csv"), dtype={"UBIGEO": str})
cruce = M.cruce_acceso_densidad(acceso_distrital, pop_completa)
cruce.to_csv(os.path.join(OUTPUTS_DIR, "cruce_acceso_densidad.csv"), index=False)
R.log(f"Correlacion (log densidad poblacional vs tiempo de acceso ponderado): "
      f"r={cruce.attrs['r_log_densidad_vs_acceso']:.3f}")


[17:29:40] Correlacion (log densidad poblacional vs tiempo de acceso ponderado): r=-0.564


## 8. Comparación entre modos (auto / a pie / bici)

Ratios de tiempo y si el establecimiento resolutivo más cercano cambia
entre auto y a pie -- la comparación que pide explícitamente la Fase 2.

In [9]:
comp_modos = pd.read_parquet(os.path.join(PROCESSED_DIR, "comparacion_modos.parquet"))
comp_modos = M.comparacion_modos(comp_modos)
comp_modos.to_csv(os.path.join(OUTPUTS_DIR, "comparacion_modos.csv"), index=False)

R.log(f"ratio foot/car mediana: {comp_modos['ratio_foot_car'].median():.1f} | "
      f"ratio bike/car mediana: {comp_modos['ratio_bike_car'].median():.1f}")
cambia_valido = comp_modos["cambia_establecimiento_auto_vs_pie"].dropna()
R.log(f"cambia de establecimiento mas cercano (auto vs pie), de los que llegan por ambos modos: "
      f"{cambia_valido.mean():.1%} ({len(cambia_valido)} puntos comparables)")


[17:29:40] ratio foot/car mediana: 10.3 | ratio bike/car mediana: 3.3


[17:29:40] cambia de establecimiento mas cercano (auto vs pie), de los que llegan por ambos modos: 11.8% (3497 puntos comparables)


## 9. Resumen final

In [10]:
print("="*70)
print("RESUMEN FASE 3")
print("="*70)
print(f"Poblacion total (muestra): {acceso['POBLACION_CP'].sum():,.0f}")
print(f"Alcanzable por auto: {acceso['ALCANZABLE'].mean():.1%}")
print(f"Gini de acceso (3 deptos): {gini_global:.3f}")
print(f"Correlacion densidad-acceso: r={cruce.attrs['r_log_densidad_vs_acceso']:.3f}")
print()
print("Archivos generados en data/outputs/:")
for f in ["tabla_acceso.csv", "cobertura_bandas_general.csv", "cobertura_bandas_departamento.csv",
          "acceso_distrital.csv", "acceso_provincial.csv", "acceso_departamental.csv",
          "brechas_criticas.csv", "gini_acceso.csv", "lorenz_acceso.csv",
          "urbano_rural.csv", "cruce_acceso_densidad.csv", "comparacion_modos.csv"]:
    p = os.path.join(OUTPUTS_DIR, f)
    print(f"  - {f}  {'[OK]' if os.path.exists(p) else '[FALTA]'}")

R.log("=== FASE 3 completa ===")


RESUMEN FASE 3
Poblacion total (muestra): 3,738,681
Alcanzable por auto: 71.2%
Gini de acceso (3 deptos): 0.633
Correlacion densidad-acceso: r=-0.564

Archivos generados en data/outputs/:
  - tabla_acceso.csv  [OK]
  - cobertura_bandas_general.csv  [OK]
  - cobertura_bandas_departamento.csv  [OK]
  - acceso_distrital.csv  [OK]
  - acceso_provincial.csv  [OK]
  - acceso_departamental.csv  [OK]
  - brechas_criticas.csv  [OK]
  - gini_acceso.csv  [OK]
  - lorenz_acceso.csv  [OK]
  - urbano_rural.csv  [OK]
  - cruce_acceso_densidad.csv  [OK]
  - comparacion_modos.csv  [OK]
[17:29:40] === FASE 3 completa ===
